# Summary Statistics Analysis

This notebook analyzes the cleaned MCSLC and SPD datasets.

The goal of this notebook is to:

- Load the cleaned datasets created in `updated_cleaned_data.ipynb`
- Summarize MCSLC response times
- Compare response times before and after the August 2024 rollout
- Summarize dispatch reasons
- Summarize dispositions
- Summarize endpoints of dispatch
- Summarize relevant SPD calls
- Save summary tables for later use

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

## 1. Set File Paths

This section sets the folder path for the cleaned data.

Because this project is currently being run in Jupyter, this notebook assumes the cleaned CSV files are in the same folder as the notebook.

In [2]:
# Use the same folder as the current Jupyter notebook
DATA_FOLDER = Path(".")

# Folder for saved summary results
RESULTS_FOLDER = Path("results")
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

print("Current working directory:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir(DATA_FOLDER))

Current working directory:
C:\Users\Phoenix M\DSiA Project

Files in this folder:
['.ipynb_checkpoints', '2015-2025 SPD Calls for Service.csv', '2015-2025 SPD Calls with Close Codes.csv', '2015-2025 SPD Responding Units.csv', 'cleaned_mcslc.csv', 'cleaned_mcslc_reasons_long.csv', 'cleaned_spd_calls.csv', 'cleaned_spd_close_codes.csv', 'cleaned_spd_relevant_calls.csv', 'cleaned_spd_units.csv', 'Clean_data.ipynb', 'data', 'Eugene_CAD_data_noloc - Copy', 'graph_boxplot_Minutes_Arrival_to_Engagement.png', 'graph_boxplot_Minutes_Dispatch_to_Arrival.png', 'graph_boxplot_Minutes_Request_to_Dispatch.png', 'graph_dispatch_endpoints_before_after.png', 'graph_dispatch_reason_categories_before_after.png', 'graph_dispatch_reason_percentages_before_after.png', 'graph_dispositions_before_after.png', 'graph_mean_bar_Minutes_Request_to_Dispatch.png', 'graph_mean_Minutes_Arrival_to_Departure_by_city_period.png', 'graph_mean_Minutes_Arrival_to_Departure_by_period.png', 'graph_mean_Minutes_Arrival_to_Enga

## 2. Load Cleaned Data

This function loads the cleaned files created by the cleaning notebook.

Before running this notebook, make sure you already ran `updated_cleaned_data.ipynb`.

In [3]:
def load_cleaned_data(data_folder=DATA_FOLDER):
    """
    Loads cleaned project datasets created by updated_cleaned_data.ipynb.
    """
    mcslc_clean = pd.read_csv(data_folder / "mcslc_clean.csv")
    reasons_long_clean = pd.read_csv(data_folder / "reasons_long_clean.csv")
    spd_calls_clean = pd.read_csv(data_folder / "spd_calls_clean.csv")
    spd_relevant = pd.read_csv(data_folder / "spd_relevant_calls_clean.csv")
    spd_close_codes_clean = pd.read_csv(data_folder / "spd_close_codes_clean.csv")
    spd_units_clean = pd.read_csv(data_folder / "spd_units_clean.csv")
    
    return (
        mcslc_clean,
        reasons_long_clean,
        spd_calls_clean,
        spd_relevant,
        spd_close_codes_clean,
        spd_units_clean
    )

## 3. Load the Cleaned Datasets

This cell calls the loading function and checks the shape of each dataset.

In [4]:
(
    mcslc_clean,
    reasons_long_clean,
    spd_calls_clean,
    spd_relevant,
    spd_close_codes_clean,
    spd_units_clean
) = load_cleaned_data()

print("MCSLC cleaned shape:", mcslc_clean.shape)
print("Reasons long shape:", reasons_long_clean.shape)
print("SPD calls cleaned shape:", spd_calls_clean.shape)
print("SPD relevant calls shape:", spd_relevant.shape)
print("SPD close codes cleaned shape:", spd_close_codes_clean.shape)
print("SPD units cleaned shape:", spd_units_clean.shape)

MCSLC cleaned shape: (4682, 21)
Reasons long shape: (9185, 9)
SPD calls cleaned shape: (54804, 11)
SPD relevant calls shape: (7288, 11)
SPD close codes cleaned shape: (98, 2)
SPD units cleaned shape: (56, 6)


## 4. Convert Date Columns

Some date columns may reload from CSV as text, so this section converts them back into datetime format.

In [5]:
# Convert MCSLC date/time columns back to datetime after loading from CSV
mcslc_datetime_cols = [
    "dispatch_request_date_and_time",
    "dispatch_date_and_time",
    "arrival_on_scene_date_and_time",
    "engagement_with_client_date_and_time",
    "mcit_departure_date_and_time"
]

for col in mcslc_datetime_cols:
    if col in mcslc_clean.columns:
        mcslc_clean[col] = pd.to_datetime(mcslc_clean[col], errors="coerce")

# Convert reasons date column if it exists
if "dispatch_request_date_and_time" in reasons_long_clean.columns:
    reasons_long_clean["dispatch_request_date_and_time"] = pd.to_datetime(
        reasons_long_clean["dispatch_request_date_and_time"],
        errors="coerce"
    )

## 5. Response Time Columns

These are the main response-time variables used in the project.

They measure different stages of the MCSLC response pipeline.

In [6]:
response_time_cols = [
    "minutes_request_to_dispatch",
    "minutes_dispatch_to_arrival",
    "minutes_arrival_to_engagement",
    "minutes_arrival_to_departure"
]

# Keep only response time columns that actually exist in the data
response_time_cols = [
    col for col in response_time_cols
    if col in mcslc_clean.columns
]

response_time_cols

['minutes_request_to_dispatch',
 'minutes_dispatch_to_arrival',
 'minutes_arrival_to_engagement',
 'minutes_arrival_to_departure']

## 6. Overall Response-Time Summary

This section summarizes response times across all MCSLC calls.

The table includes:

- Count
- Mean
- Median
- Standard deviation
- Minimum
- Maximum

In [7]:
overall_response_summary = mcslc_clean[response_time_cols].describe().T

overall_response_summary

,count,mean,std,min,25%,50%,75%,max
minutes_request_to_dispatch,4343.0,21.375086,25.391028,0.0,6.0,13.0,26.0,180.0
minutes_dispatch_to_arrival,4256.0,16.129699,13.539362,0.0,8.0,14.0,21.0,156.0
minutes_arrival_to_engagement,3466.0,1.965089,3.201586,0.0,0.0,1.0,2.0,30.0
minutes_arrival_to_departure,4046.0,29.019525,25.577413,0.0,10.0,22.0,40.0,178.0


## 7. Response-Time Summary by Rollout Period

This section compares response times before and after the August 2024 MCSLC rollout.

This directly connects to the project question:

How did MCSLC response times change following the rollout?

In [8]:
response_summary_by_rollout = (
    mcslc_clean
    .groupby("rollout_period")[response_time_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

response_summary_by_rollout

minutes_request_to_dispatch                                    \
                                     count       mean median        std  min   
rollout_period                                                                 
After Rollout                         4343  21.375086   13.0  25.391028  0.0   

                      minutes_dispatch_to_arrival                    \
                  max                       count       mean median   
rollout_period                                                        
After Rollout   180.0                        4256  16.129699   14.0   

                           ... minutes_arrival_to_engagement                 \
                      std  ...                        median       std  min   
rollout_period             ...                                                
After Rollout   13.539362  ...                           1.0  3.201586  0.0   

                     minutes_arrival_to_departure                    \
                 max                        count       mean median   
rollout_period                                                        
After Rollout   30.0                         4046  29.019525   22.0   

                                       
                      std  min    max  
rollout_period                         
After Rollout   25.577413  0.0  178.0  

[1 rows x 24 columns]

## 8. Cleaner Rollout Comparison Table

This section creates an easier-to-read before/after table using the mean and median response time for each stage.

In [9]:
rollout_mean_median = (
    mcslc_clean
    .groupby("rollout_period")[response_time_cols]
    .agg(["mean", "median"])
)

rollout_mean_median

minutes_request_to_dispatch        minutes_dispatch_to_arrival  \
                                      mean median                        mean   
rollout_period                                                                  
After Rollout                    21.375086   13.0                   16.129699   

                      minutes_arrival_to_engagement         \
               median                          mean median   
rollout_period                                               
After Rollout    14.0                      1.965089    1.0   

               minutes_arrival_to_departure         
                                       mean median  
rollout_period                                      
After Rollout                     29.019525   22.0

## 9. Difference Between Before and After Rollout

This section calculates the average difference between the before-rollout and after-rollout periods.

A positive number means the response time was higher after rollout.

A negative number means the response time was lower after rollout.

In [10]:
rollout_means = (
    mcslc_clean
    .groupby("rollout_period")[response_time_cols]
    .mean()
)

rollout_means

,minutes_request_to_dispatch,minutes_dispatch_to_arrival,minutes_arrival_to_engagement,minutes_arrival_to_departure
rollout_period,,,,
After Rollout,21.375086,16.129699,1.965089,29.019525


In [11]:
# Calculate after-before difference if both groups exist
if "Before Rollout" in rollout_means.index and "After Rollout" in rollout_means.index:
    rollout_mean_difference = (
        rollout_means.loc["After Rollout"] 
        - rollout_means.loc["Before Rollout"]
    )
    
    rollout_mean_difference = rollout_mean_difference.to_frame(name="after_minus_before_mean")
else:
    rollout_mean_difference = pd.DataFrame()

rollout_mean_difference

""


## 10. Monthly Response-Time Summary

This section summarizes average response times by month.

This helps show whether response times changed gradually over time.

In [12]:
monthly_response_summary = (
    mcslc_clean
    .groupby("dispatch_month")[response_time_cols]
    .mean()
    .reset_index()
)

monthly_response_summary

,dispatch_month,minutes_request_to_dispatch,minutes_dispatch_to_arrival,minutes_arrival_to_engagement,minutes_arrival_to_departure
0,2024-08,16.761194,20.312500,2.203704,36.687500
1,2024-09,26.025974,21.979310,2.324786,34.157534
2,2024-10,11.041667,27.458333,3.238095,32.043478
3,2025-01,13.979310,18.063380,2.322034,34.419580
4,2025-02,15.083333,16.356164,2.619048,31.358621
5,2025-03,18.263158,18.260638,2.245283,37.608696
6,2025-04,16.118705,15.669065,1.798283,29.562724
7,2025-05,23.278826,15.697917,1.887006,25.476395
8,2025-06,23.243993,15.980952,1.819222,25.862069
9,2025-07,24.825123,16.948718,1.529052,26.945799


## 11. Dispatch Endpoint Summary

This section counts the endpoint of dispatch.

This is useful for seeing where calls were routed.

In [13]:
if "end_point_of_dispatch" in mcslc_clean.columns:
    endpoint_summary = (
        mcslc_clean["end_point_of_dispatch"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    endpoint_summary.columns = ["end_point_of_dispatch", "count"]
else:
    endpoint_summary = pd.DataFrame()

endpoint_summary

,end_point_of_dispatch,count
0,NaN,1592
1,Engaged Client,1107
2,Unable to Locate,649
3,Refused,532
4,Unable to locate client,285
5,Engaged client,228
6,Cancelled,215
7,Client declined to engage,27
8,No contact due to safety concern,19
9,Client declined to engage,19


## 12. Dispatch Endpoint by Rollout Period

This section compares dispatch endpoints before and after rollout.

In [14]:
if "end_point_of_dispatch" in mcslc_clean.columns and "rollout_period" in mcslc_clean.columns:
    endpoint_by_rollout = pd.crosstab(
        mcslc_clean["end_point_of_dispatch"],
        mcslc_clean["rollout_period"]
    )
else:
    endpoint_by_rollout = pd.DataFrame()

endpoint_by_rollout

rollout_period,After Rollout
end_point_of_dispatch,
Cancelled,215
Client declined to engage,19
Client declined to engage,27
Dispatch canceled before arrival,4
Engaged Client,1107
Engaged client,228
No contact due to safety concern,19
Other,5
Refused,532


## 13. Disposition Summary

This section counts the most common MCSLC dispositions.

In [15]:
if "disposition" in mcslc_clean.columns:
    disposition_summary = (
        mcslc_clean["disposition"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    disposition_summary.columns = ["disposition", "count"]
else:
    disposition_summary = pd.DataFrame()

disposition_summary

,disposition,count
0,Remained in community,3750
1,Emergency Department,652
2,Other,230
3,Sobering/Detox Facility,26
4,Arrest,16
5,Crisis walk-in Center,3
6,Respite,3
7,Sobering or Detox Facility,2


## 14. Disposition by Rollout Period

This section compares dispositions before and after rollout.

In [16]:
if "disposition" in mcslc_clean.columns and "rollout_period" in mcslc_clean.columns:
    disposition_by_rollout = pd.crosstab(
        mcslc_clean["disposition"],
        mcslc_clean["rollout_period"]
    )
else:
    disposition_by_rollout = pd.DataFrame()

disposition_by_rollout

rollout_period,After Rollout
disposition,
Arrest,16
Crisis walk-in Center,3
Emergency Department,652
Other,230
Remained in community,3750
Respite,3
Sobering or Detox Facility,2
Sobering/Detox Facility,26


## 15. Dispatch Reasons Summary

This section counts the most common reasons for dispatch using the long-format reasons dataset.

In [17]:
if "reason_for_dispatch" in reasons_long_clean.columns:
    reason_summary = (
        reasons_long_clean["reason_for_dispatch"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    reason_summary.columns = ["reason_for_dispatch", "count"]
else:
    reason_summary = pd.DataFrame()

reason_summary.head(20)

,reason_for_dispatch,count
0,Agitation or disruptive behavior,2098
1,Disorganized behavior,1520
2,Difficulty functioning,1335
3,Needing social/mental health services,1066
4,Suicidality or suicide attempt,1015
5,Harm/risk of harm to self/others/property,867
6,Adult Social/interpersonal problems,356
7,Paranoia,340
8,Substance use,269
9,Other,96


## 16. Dispatch Reasons by Rollout Period

This section compares dispatch reasons before and after rollout.

In [19]:
if "reason_for_dispatch" in reasons_long_clean.columns and "rollout_period" in reasons_long_clean.columns:
    reason_by_rollout = pd.crosstab(
        reasons_long_clean["reason_for_dispatch"],
        reasons_long_clean["rollout_period"]
    )
else:
    reason_by_rollout = pd.DataFrame()

reason_by_rollout

rollout_period,After Rollout
reason_for_dispatch,
Adult Social/interpersonal problems,356
Agitation or disruptive behavior,2098
Concerns about treatment engagement,78
Difficulty functioning,1335
Disorganized behavior,1520
Harm/risk of harm to self/others/property,867
Needing social/mental health services,1066
Other,96
Paranoia,340


## 17. Top Dispatch Reasons Before and After Rollout

This section creates a cleaner table showing the top reasons during each rollout period.

In [20]:
if "reason_for_dispatch" in reasons_long_clean.columns and "rollout_period" in reasons_long_clean.columns:
    top_reasons_by_rollout = (
        reasons_long_clean
        .groupby(["rollout_period", "reason_for_dispatch"])
        .size()
        .reset_index(name="count")
        .sort_values(["rollout_period", "count"], ascending=[True, False])
    )
else:
    top_reasons_by_rollout = pd.DataFrame()

top_reasons_by_rollout.head(30)

,rollout_period,reason_for_dispatch,count
1,After Rollout,Agitation or disruptive behavior,2098
4,After Rollout,Disorganized behavior,1520
3,After Rollout,Difficulty functioning,1335
6,After Rollout,Needing social/mental health services,1066
12,After Rollout,Suicidality or suicide attempt,1015
5,After Rollout,Harm/risk of harm to self/others/property,867
0,After Rollout,Adult Social/interpersonal problems,356
8,After Rollout,Paranoia,340
11,After Rollout,Substance use,269
7,After Rollout,Other,96


## 18. SPD Relevant Calls Summary

This section summarizes the filtered SPD calls that may be relevant to crisis response, behavioral health, welfare checks, medical calls, or similar call types.

In [21]:
print("Total SPD calls:", len(spd_calls_clean))
print("Relevant SPD calls:", len(spd_relevant))

if len(spd_calls_clean) > 0:
    relevant_percent = (len(spd_relevant) / len(spd_calls_clean)) * 100
    print("Percent relevant:", round(relevant_percent, 2), "%")

Total SPD calls: 54804
Relevant SPD calls: 7288
Percent relevant: 13.3 %


## 19. SPD Initial Call Type Summary

This section counts the most common initial call types among the relevant SPD calls.

In [22]:
if "initial_call_type" in spd_relevant.columns:
    spd_initial_call_summary = (
        spd_relevant["initial_call_type"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    spd_initial_call_summary.columns = ["initial_call_type", "count"]
else:
    spd_initial_call_summary = pd.DataFrame()

spd_initial_call_summary.head(20)

,initial_call_type,count
0,CHKWLF,2079
1,SUSPSU,1204
2,DSRDSU,828
3,SUISUB,366
4,ASTPUB,327
5,INTXSU,318
6,DSRDJU,245
7,ASTOUT,244
8,LOCSUB,202
9,SUBJDO,175


## 20. SPD Final Call Type Summary

This section counts the most common final call types among the relevant SPD calls.

In [23]:
if "final_call_type" in spd_relevant.columns:
    spd_final_call_summary = (
        spd_relevant["final_call_type"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    spd_final_call_summary.columns = ["final_call_type", "count"]
else:
    spd_final_call_summary = pd.DataFrame()

spd_final_call_summary.head(20)

,final_call_type,count
0,CHECK WELFARE,2252
1,SUSPICIOUS SUBJECT,1219
2,DISORDERLY SUBJECT,867
3,ASSIST PUBLIC- POLICE,348
4,INTOXICATED SUBJECT,339
5,SUICIDAL SUBJECT,330
6,ASSIST OUTSIDE AGENCY,296
7,DISORDERLY JUVENILES,260
8,LOCATION WANTED SUBJECT,232
9,SUBJECT DOWN,169


## 21. SPD Priority Summary

This section counts priority levels among relevant SPD calls.

In [24]:
if "priority" in spd_relevant.columns:
    spd_priority_summary = (
        spd_relevant["priority"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    spd_priority_summary.columns = ["priority", "count"]
else:
    spd_priority_summary = pd.DataFrame()

spd_priority_summary

,priority,count
0,3,3263
1,4,3068
2,7,634
3,5,137
4,2,44
5,9,41
6,6,35
7,8,29
8,,26
9,P,6


## 22. SPD Responding Agency Summary

This section counts responding agencies among relevant SPD calls.

In [25]:
if "responding_agency" in spd_relevant.columns:
    spd_agency_summary = (
        spd_relevant["responding_agency"]
        .value_counts(dropna=False)
        .reset_index()
    )
    
    spd_agency_summary.columns = ["responding_agency", "count"]
else:
    spd_agency_summary = pd.DataFrame()

spd_agency_summary

,responding_agency,count
0,SPD,7288


## 23. Save Summary Tables

This section saves the summary tables as CSV files.

These can be used later in the final report, README, or visualizations.

In [26]:
# Save MCSLC summaries
overall_response_summary.to_csv(RESULTS_FOLDER / "overall_response_summary.csv")
response_summary_by_rollout.to_csv(RESULTS_FOLDER / "response_summary_by_rollout.csv")
rollout_mean_median.to_csv(RESULTS_FOLDER / "rollout_mean_median.csv")
rollout_mean_difference.to_csv(RESULTS_FOLDER / "rollout_mean_difference.csv")
monthly_response_summary.to_csv(RESULTS_FOLDER / "monthly_response_summary.csv", index=False)

# Save categorical summaries
endpoint_summary.to_csv(RESULTS_FOLDER / "endpoint_summary.csv", index=False)
endpoint_by_rollout.to_csv(RESULTS_FOLDER / "endpoint_by_rollout.csv")
disposition_summary.to_csv(RESULTS_FOLDER / "disposition_summary.csv", index=False)
disposition_by_rollout.to_csv(RESULTS_FOLDER / "disposition_by_rollout.csv")
reason_summary.to_csv(RESULTS_FOLDER / "reason_summary.csv", index=False)
reason_by_rollout.to_csv(RESULTS_FOLDER / "reason_by_rollout.csv")
top_reasons_by_rollout.to_csv(RESULTS_FOLDER / "top_reasons_by_rollout.csv", index=False)

# Save SPD summaries
spd_initial_call_summary.to_csv(RESULTS_FOLDER / "spd_initial_call_summary.csv", index=False)
spd_final_call_summary.to_csv(RESULTS_FOLDER / "spd_final_call_summary.csv", index=False)
spd_priority_summary.to_csv(RESULTS_FOLDER / "spd_priority_summary.csv", index=False)
spd_agency_summary.to_csv(RESULTS_FOLDER / "spd_agency_summary.csv", index=False)

print("Summary tables saved to:", RESULTS_FOLDER)

Summary tables saved to: results


## 24. Final Analysis Notes

This notebook created the main summary statistics for the project.

The next notebook should focus on graphing these results one by one.